# Thư viện và dữ liệu

In [1]:
#thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from sklearn.model_selection import ParameterGrid
import warnings
warnings.filterwarnings('ignore')

In [1]:
#tải dữ liệu
df = pd.read_csv('../tnbike_data.csv', parse_dates=['ds'])
df = df.rename(columns={'revenue': 'y'})
df.head()

In [1]:
#biến ngày
df['ds'] = pd.to_datetime(df['ds'])
df.ds

0,2025-01-01
1,2025-02-01


# Ngày lễ

In [1]:
#Tết Nguyên Đán
dates_tet = pd.to_datetime(df[df.is_tet == 1].ds)
tet = pd.DataFrame({'holiday': 'tet_nguyen_dan',
                    'ds': dates_tet,
                    'lower_window': -7,
                    'upper_window': 7})

In [1]:
#Ngày Quốc tế Thiếu nhi 1/6
dates_thieu_nhi = pd.to_datetime(df[df.is_thieu_nhi == 1].ds)
thieu_nhi = pd.DataFrame({'holiday': 'ngay_thieu_nhi',
                          'ds': dates_thieu_nhi,
                          'lower_window': -14,
                          'upper_window': 3})

In [1]:
thieu_nhi

,holiday,ds,lower_window,upper_window
4,ngay_thieu_nhi,2025-05-01,-14,3


In [1]:
#gộp ngày lễ
holidays = pd.concat([tet, thieu_nhi])
holidays

,holiday,ds
0,tet_nguyen_dan,2025-01-01
4,ngay_thieu_nhi,2025-05-01


In [1]:
#bỏ cột ngày lễ khỏi df chính
df = df.drop(['is_tet', 'is_thieu_nhi'], axis=1)

In [1]:
#tách train
training = df.iloc[:-3, :]
future_df = df.iloc[-3:, :]

# Lưới tham số Prophet

In [1]:
#định nghĩa lưới tham số
param_grid = {'changepoint_prior_scale': [0.01, 0.05, 0.1, 0.5],
              'holidays_prior_scale':    [0.01, 1.0, 10.0],
              'seasonality_prior_scale': [0.01, 1.0, 10.0],
              'seasonality_mode':        ['additive', 'multiplicative']}
grid = list(ParameterGrid(param_grid))
print(f'Tổng tổ hợp: {len(grid)}')

Tổng tổ hợp: 72


In [1]:
#chạy cross-validation để tìm tham số tốt nhất
rmse = []
for params in grid:
    m = Prophet(holidays=holidays,
                seasonality_mode=params['seasonality_mode'],
                seasonality_prior_scale=params['seasonality_prior_scale'],
                holidays_prior_scale=params['holidays_prior_scale'],
                changepoint_prior_scale=params['changepoint_prior_scale'],
                yearly_seasonality=False,
                weekly_seasonality=False,
                daily_seasonality=False)
    m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    m.fit(training)
    df_cv = cross_validation(m, horizon='90 days', period='30 days', initial='90 days')
    df_p  = performance_metrics(df_cv, rolling_window=1)
    rmse.append(df_p['rmse'].values[0])
print('Tinh chỉnh xong!')

Tinh chỉnh xong!


# Kết quả tinh chỉnh

In [1]:
#xem bảng kết quả
tuning_results = pd.DataFrame(grid)
tuning_results['rmse'] = rmse
tuning_results

changepoint_prior_scale,holidays_prior_scale,seasonality_prior_scale,seasonality_mode,rmse
0.05,10.0,10.0,multiplicative,1.23e+09


In [1]:
#tham số tốt nhất
best_params = tuning_results[tuning_results.rmse == tuning_results.rmse.min()].transpose()
best_params

,0
changepoint_prior_scale,0.05
holidays_prior_scale,10.0
seasonality_prior_scale,10.0
seasonality_mode,multiplicative


In [1]:
#xuất tham số tốt nhất
best_params.to_csv('../Forecasting Product/best_params_prophet.csv')
print('✅ Đã lưu best_params_prophet.csv')

✅ Đã lưu best_params_prophet.csv
